In [1]:
import pandas as pd
import numpy as np
import joblib
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

df = pd.read_csv("../data/processed/cnc_features.csv")

X = df.drop(columns="defect")   # features 
y = df["defect"]                # target 

print(X.shape, y.shape)

(2000, 12) (2000,)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print("Train:", X_train.shape, "| Test:", X_test.shape)
print("Train defect %:", round(y_train.mean() * 100, 1))
print("Test defect %: ", round(y_test.mean() * 100, 1))

Train: (1600, 12) | Test: (400, 12)
Train defect %: 25.1
Test defect %:  25.2


In [3]:
ratio = (y_train == 0).sum() / (y_train == 1).sum()
print("Good : Defective ratio =", round(ratio, 2))

models = {
    "Logistic Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42))
    ]),
    "Random Forest": RandomForestClassifier(
        n_estimators=200, class_weight="balanced", random_state=42
    ),
    "XGBoost": XGBClassifier(
        n_estimators=200, learning_rate=0.1, max_depth=4,
        scale_pos_weight=ratio, eval_metric="logloss", random_state=42
    ),
}

Good : Defective ratio = 2.98


In [4]:
results = []

for name, model in models.items():
    model.fit(X_train, y_train)                   
    y_pred = model.predict(X_test)               
    y_prob = model.predict_proba(X_test)[:, 1]    

    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1": f1_score(y_test, y_pred),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
    })

results_df = pd.DataFrame(results).set_index("Model").round(3)
results_df

,Accuracy,Precision,Recall,F1,ROC-AUC
Model,,,,,
Logistic Regression,0.860,0.680,0.842,0.752,0.944
Random Forest,0.872,0.727,0.792,0.758,0.933
XGBoost,0.878,0.750,0.772,0.761,0.930


In [5]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

for name, model in models.items():
    scores = cross_val_score(model, X_train, y_train, cv=cv, scoring="f1")
    print(f"{name:20s} F1: {scores.mean():.3f} (+/- {scores.std():.3f})")

Logistic Regression  F1: 0.719 (+/- 0.023)
Random Forest        F1: 0.705 (+/- 0.028)
XGBoost              F1: 0.682 (+/- 0.020)


In [6]:
joblib.dump(models["Logistic Regression"], "../models/logistic_regression.joblib")
joblib.dump(models["Random Forest"], "../models/random_forest.joblib")
joblib.dump(models["XGBoost"], "../models/xgboost.joblib")

X_test.to_csv("../data/processed/X_test.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)
print("Saved ✅")

Saved ✅


In [1]:
import joblib
import pandas as pd

X_test = pd.read_csv("../data/processed/X_test.csv")
joblib.dump(list(X_test.columns), "../models/feature_columns.joblib")
print(list(X_test.columns))

['spindle_speed_rpm', 'feed_rate_mm_min', 'cutting_temp_c', 'vibration_mm_s', 'tool_wear_min', 'coolant_flow_lpm', 'cutting_intensity', 'wear_x_vibration', 'heat_per_coolant', 'vibration_per_speed', 'material_type_Steel', 'material_type_Titanium']
